# Bank Loan Portfolio — Dashboard Training Notebook

This notebook walks through the full journey from **raw data** to a
**web dashboard**, using synthetic (fake) bank loan data so it's safe to
practice on.

**What you'll learn in this notebook:**
1. Loading data with pandas
2. Exploring and cleaning it
3. Calculating banking KPIs (loan book, default rate, NPL ratio)
4. Static charts with Matplotlib
5. Interactive charts with Plotly
6. How this same logic becomes a live Streamlit dashboard
7. How to run the dashboard and what "sharing" it actually means

> **Note:** All data in `data/bank_loan_portfolio.csv` is randomly generated
> and does not represent real NCBA customers, branches, or figures.


## 1. Load the data

We use `pandas` — the standard Python library for working with tabular data
(think of it as Excel, but scriptable and much faster for large datasets).

`pd.read_csv()` reads a CSV file into a **DataFrame** — pandas' name for a
table with rows and columns.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/bank_loan_portfolio.csv", parse_dates=["Disbursement_Date"])

# parse_dates tells pandas to treat that column as actual dates,
# not plain text — this matters later for time-based charts.

df.head()  # shows the first 5 rows so you can eyeball the structure


In [ ]:
# .shape tells you (number of rows, number of columns)
df.shape


In [ ]:
# .info() is a quick health check: column names, data types, and
# how many non-empty values are in each column
df.info()


## 2. Explore and understand the columns

| Column | Meaning |
|---|---|
| `Customer_ID` | Unique customer identifier |
| `Branch` / `Region` | Where the loan was booked |
| `Product` | Loan type (Motor, SME, Personal, Mortgage, Asset Finance, Overdraft) |
| `Customer_Segment` | Retail / SME / Corporate |
| `Loan_Officer` | Officer who booked the loan |
| `Disbursement_Date` | When the loan was disbursed |
| `Tenure_Months` | Loan term in months |
| `Interest_Rate` | Annual interest rate (decimal, e.g. 0.15 = 15%) |
| `Disbursed_Amount` | Original loan amount |
| `Outstanding_Balance` | Amount still owed today |
| `DPD` | Days Past Due — how overdue the loan is |
| `IFRS9_Stage` | Risk classification bucket based on DPD |


In [ ]:
# Quick look at unique values in key categorical columns
print("Branches:", df['Branch'].unique())
print()
print("Products:", df['Product'].unique())
print()
print("IFRS9 Stages:", df['IFRS9_Stage'].unique())


In [ ]:
# .describe() gives quick summary statistics for numeric columns
# (count, mean, std deviation, min, max, quartiles)
df[['Disbursed_Amount', 'Outstanding_Balance', 'Interest_Rate', 'DPD']].describe()


In [ ]:
# Check for missing values in each column
df.isnull().sum()


## 3. Calculate banking KPIs

This is the "analysis" layer — turning raw rows into the numbers a manager
actually wants to see. These are the same calculations that will later
become the KPI cards on the dashboard.


In [ ]:
total_book = df['Outstanding_Balance'].sum()
print(f"Total Outstanding Loan Book: KES {total_book:,.0f}")


In [ ]:
# Default rate: proportion of loans more than 30 days past due
default_rate = (df['DPD'] > 30).sum() / len(df)
print(f"Default Rate (>30 DPD): {default_rate:.1%}")


In [ ]:
# NPL ratio using the IFRS9_Stage column (Stage 3 = Non-Performing)
npl_balance = df.loc[df['IFRS9_Stage'] == 'Stage 3 - Non-Performing', 'Outstanding_Balance'].sum()
npl_ratio = npl_balance / total_book
print(f"NPL Balance: KES {npl_balance:,.0f}")
print(f"NPL Ratio: {npl_ratio:.1%}")


In [ ]:
# Portfolio breakdown by IFRS9 stage
stage_summary = df.groupby('IFRS9_Stage').agg(
    Number_of_Loans=('Customer_ID', 'count'),
    Outstanding_Balance=('Outstanding_Balance', 'sum')
).reset_index()

stage_summary


In [ ]:
# Which branch has the largest book?
branch_summary = df.groupby('Branch')['Outstanding_Balance'].sum().sort_values(ascending=False)
branch_summary


## 4. Static charts with Matplotlib

Matplotlib is the foundational Python plotting library. Charts are **static
images** — good for reports and slides, but not interactive.


In [ ]:
import matplotlib.pyplot as plt

product_summary = df.groupby('Product')['Outstanding_Balance'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
product_summary.plot(kind='bar', color='#1f4e79')
plt.title('Outstanding Balance by Product')
plt.ylabel('Outstanding Balance (KES)')
plt.xlabel('Product')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# A pie chart of portfolio mix by segment
segment_summary = df.groupby('Customer_Segment')['Outstanding_Balance'].sum()

plt.figure(figsize=(6, 6))
plt.pie(segment_summary, labels=segment_summary.index, autopct='%1.1f%%', startangle=90)
plt.title('Portfolio Mix by Customer Segment')
plt.show()


## 5. Interactive charts with Plotly

Plotly charts run in a browser and let the viewer **hover, zoom, and
filter** — this is the same charting library the dashboard app uses, so
what you build here is directly reusable.


In [ ]:
import plotly.express as px

fig = px.bar(
    product_summary.reset_index(),
    x='Product',
    y='Outstanding_Balance',
    color='Product',
    title='Outstanding Balance by Product (Interactive)',
    text_auto='.2s'
)
fig.show()


In [ ]:
# Trend of disbursements over time
trend = (
    df.set_index('Disbursement_Date')
      .resample('ME')['Disbursed_Amount']
      .sum()
      .reset_index()
)

fig = px.line(
    trend,
    x='Disbursement_Date',
    y='Disbursed_Amount',
    title='Monthly Disbursement Trend',
    markers=True
)
fig.show()


In [ ]:
# DPD distribution by branch — a box plot is useful for spotting
# which branches have wider risk spread
fig = px.box(
    df,
    x='Branch',
    y='DPD',
    title='DPD Distribution by Branch',
    color='Branch'
)
fig.show()


## 6. From notebook to dashboard: what changes?

Everything you did above — loading data, calculating KPIs, building
Plotly charts — is **exactly the same code** used inside `dashboard.py`
(included in this package). The difference is just the wrapper:

| In this notebook | In the dashboard (`dashboard.py`) |
|---|---|
| `df.head()` shows a table once | `st.dataframe(df)` shows a live, scrollable, sortable table |
| `print(f"...")` prints a number once | `st.metric("Label", value)` shows it as a styled KPI card |
| `fig.show()` opens chart in this notebook | `st.plotly_chart(fig)` embeds it in the web page |
| You manually re-run cells to filter | `st.sidebar.multiselect(...)` lets the *user* filter live, in their browser |

Streamlit takes your script and **reruns it top to bottom every time a user
interacts with a filter/button** — that's the whole trick. There's no
separate "frontend" you have to build; the Python script *is* the web app.


## 7. Running the dashboard

The dashboard code lives in `dashboard.py` in this same folder. This
notebook is for **learning/exploring** — the dashboard is for **viewing and
sharing**. To launch it:

1. Open a terminal (VS Code terminal, or Anaconda Prompt) in this folder.
2. Make sure the packages are installed:
   ```
   pip install -r requirements.txt
   ```
3. Run:
   ```
   streamlit run dashboard.py
   ```
4. **What happens next:**
   - Streamlit starts a small local web server on your machine.
   - Your terminal will print something like:
     ```
     Local URL: http://localhost:8501
     Network URL: http://192.168.1.15:8501
     ```
   - Your **default web browser opens automatically** to that address.
   - If it doesn't open automatically, copy the `Local URL` and paste it
     into your browser manually.
5. To stop the dashboard, go back to the terminal and press `Ctrl + C`.

### What actually gets "shared" with other users

Nothing is uploaded anywhere by default — `streamlit run` only starts a
server on **your own laptop**. What "sharing" means depends on the option:

- **Same office network:** you send colleagues the `Network URL` printed in
  your terminal (e.g. `http://192.168.1.15:8501`). They open it in their own
  browser, on the same Wi-Fi/LAN. Your laptop must stay on and the app must
  keep running — close it and the link stops working for everyone.
- **Streamlit Community Cloud:** you push your project to GitHub and deploy
  it at share.streamlit.io — Streamlit hosts it for you and gives a public
  URL that works from anywhere, anytime, without your laptop running. Only
  suitable for **non-sensitive/demo data**, since it's on public
  infrastructure.
- **Internal company server:** for real banking data, IT sets up an internal
  server/VM that runs the app persistently, reachable only within NCBA's
  network/VPN. This is the appropriate option once you move past
  training/demo data.

For this training package, stick to running it locally on your own machine
and viewing it yourself at `localhost:8501` — that's the best way to learn
before thinking about sharing.
